In [0]:
CATALOG  = "workspace"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.silver.events (
              event_id STRING NOT NULL,
              event_type STRING,
              created_at TIMESTAMP,
              event_date DATE,
              event_hour INT,
              actor_id BIGINT,
              actor_login STRING,
              is_bot BOOLEAN,
              repo_id BIGINT,
              repo_name STRING,
              org_id BIGINT,
              org_login STRING,
              action STRING,
              push_size INT,
              payload STRING,
              _ingest_file STRING,
              _ingest_ts TIMESTAMP
          ) PARTITIONED BY (event_date)
          """)

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.silver.events_quarantine(
              raw STRING,
              dq_reason STRING,
              _ingest_file STRING,
              _quaratined_ts TIMESTAMP
          )""")

In [0]:
spark.sql("SHOW TABLES IN workspace.silver").show()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.silver.events")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.silver.events_quarantine")
dbutils.fs.rm(f"/Volumes/{CATALOG}/bronze/checkpoints/silver_events", recurse=True)

In [0]:
from src.writers.silver_writer import make_upsert

CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/silver_events"

(
    spark.readStream.table(f"{CATALOG}.bronze.events_raw")
    .writeStream
    .foreachBatch(make_upsert(CATALOG))
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)


In [0]:
spark.sql(f"SELECT count(*) AS silver_rows FROM {CATALOG}.silver.events").show()
spark.sql(f"SELECT count(DISTINCT id) AS bronze_distinct FROM {CATALOG}.bronze.events_raw").show()

In [0]:
spark.sql(f"SELECT count(*) AS silver_rows FROM {CATALOG}.silver.events").show()

In [0]:
spark.sql(f"SELECT dq_reason, count(*) c FROM {CATALOG}.silver.events_quarantine GROUP BY 1 ORDER BY c DESC").show()

In [0]:
spark.sql(f"""
    SELECT b.id, b.created_at, b._ingest_file, count(*) AS copies
    FROM {CATALOG}.bronze.events_raw b
    LEFT ANTI JOIN {CATALOG}.silver.events s
    ON b.id = s.event_id
    GROUP BY b.id, b.created_at, b._ingest_file
""").show(20, truncate=False)

In [0]:
spark.sql(f"SELECT count(*) AS null_ids FROM {CATALOG}.bronze.events_raw WHERE id IS NULL").show()

In [0]:
spark.sql(f"SELECT event_type, count(*) c FROM {CATALOG}.silver.events GROUP BY 1 ORDER BY c DESC").show()
spark.sql(f"SELECT is_bot, count(*) FROM {CATALOG}.silver.events GROUP BY 1").show()